# Day 3 — Solution: OHLC, Volume, Liquidity

In [ ]:
import os, sys, pathlib
root = pathlib.Path.cwd()
for _ in range(6):
    if (root / "qrc").is_dir():
        break
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import numpy as np
import pandas as pd
DATA_SOURCE = os.environ.get("QRC_DATA", "real")
from qrc.data import get_prices
from qrc.synth import synthetic_prices

## E1 — three liquidity tiers

In [ ]:
rng = np.random.default_rng(2)
n = 1000
spec = [("ETF", 500.0, 2e6, 0.008), ("LargeStock", 80.0, 6e5, 0.014),
        ("SmallStock", 12.0, 1.6e5, 0.022)]   # (name, price, ~shares, vol)
data = {}
for name, p0, sh, vol in spec:
    r = rng.normal(0.0003, vol, n)
    px = p0 * np.cumprod(1 + r)
    volume = sh * np.exp(rng.normal(0, 0.5, n))      # noise around ADV
    data[name] = pd.DataFrame({"px": px, "vol": volume})
dv = {k: v.px * v.vol for k, v in data.items()}
for k in data:
    r = data[k].px.pct_change()
    amihud = (r.abs() / dv[k]).mean() * 1e8      # bp per $1M
    print(f"{k:11s}: median DV ${dv[k].median()/1e6:7.1f}M | Amihud {amihud:6.2f} bp/$1M")

**Expected pattern.** DV spans $2M → $1B (500×); Amihud spans ~0.01 →
~50 bp/$1M (5000×): **illiquidity scales with the SQUARE of relative
size** (same move, less money) — which is why small-cap trading costs
are not a linear but a quadratic problem, and why capacity talk is
always in participation rates, never dollars-flat.

## E2 — the screens

In [ ]:
for k, v in data.items():
    px_ok = v.px.iloc[-1] > 5
    adv_ok = dv[k].rolling(20).mean().iloc[-1] > 1e6
    print(f"{k:11s}: price screen {'PASS' if px_ok else 'FAIL'} | "
          f"ADV screen {'PASS' if adv_ok else 'FAIL'}")

ETF and LargeStock pass both; SmallStock passes price but its DV
(~$2M) is borderline at the $1M ADV screen — pass by median, fail on
bad days. **Disposition policy (exemplar): screens are universe
DEFINITIONS, not data hygiene — the borderline name enters a
"monitored satellite" bucket with its own cost model (35bp+) and a
hard cap (≤2% of book), and the study reports results with and
without the bucket.** The binary include/exclude hides the one
decision that matters (does the signal survive realistic costs at the
name's own liquidity?).

## E3 — volume-vol

In [ ]:
for k, v in data.items():
    r = v.px.pct_change()
    c = r.abs().corr(np.log(dv[k]))
    print(f"{k:11s}: corr(|r|, log DV) = {c:+.2f}   (volume is pure noise)")

# now EMBED the relation (Cont fact 6) and check the estimator recovers it
for k, v in data.items():
    r = v.px.pct_change()
    v["vol_linked"] = v["vol"] * (1 + 40*r.abs())   # big moves -> big volume
    c = r.abs().corr(np.log(v["vol_linked"] * v["px"]))
    print(f"{k:11s}: corr with embedded link  = {c:+.2f}")

**Expected:** pure-noise volume gives correlations within ±0.05 of
zero — the estimator's null behavior; with the link embedded
(volume × (1 + 40·|r|)), the correlation recovers +0.2 to +0.4, the
real-market magnitude (Cont's fact 6: volume-volatility correlation).
A NEGATIVE correlation in real data is a mechanic worth knowing:
high-volume days with SMALL moves = absorption (a large passive
buyer absorbed by liquidity) — the signature of institutional
accumulation and of index rebalances.

## E4 — capacity arithmetic

In [ ]:
aum = 500e6
for k, v in data.items():
    adv20 = dv[k].rolling(20).mean().iloc[-1]
    cap_pos = 0.04 * adv20
    print(f"{k:11s}: max position at 4% of ADV = ${cap_pos/1e6:8.1f}M")
print(f"sum of caps: ${sum(0.04*dv[k].rolling(20).mean().iloc[-1] for k in data)/1e6:.0f}M "
      f"vs AUM ${aum/1e6:.0f}M")

**Expected Reasoning.** Caps ≈ $40M / $2.4M / $80k. With a 3-name
universe the book cannot diversify into the small name at all — the
strategy is 95%+ ETF+large, and its "cross-sectional" character is
fiction. Scaled to 50 names: the question is never "can $500M trade
this strategy" but "how many names have ADV ≥ $250M at the 4% cap" —
**capacity is the count of liquid names times the cap, and the small-
cap signal content is capacity-weighted to zero.** This is the quiet
truth under every "significant in small caps" paper.

## E5 — the $5 screen paragraph (exemplar)

The sentence hides a universe definition dressed as data hygiene.
Excluding sub-$5 stocks deletes: distressed names in their final
months (where short-leg reversal signals are strongest), post-crash
recoverers (momentum's best longs), and entire small-cap cohorts —
biasing measured anomalies toward their large-cap, weaker versions.
Direction: reversal (which lives in overreaction, concentrated in
low-price names) is understated; momentum-crash studies lose the
crash victims; the size premium is measured on the survivors of its
own screen. The honest paper reports the boundary: results at $1/$3/
$5/$10 cutoffs, and the sentence "our claims apply to stocks above
$X, where the effects are smaller" — which is a different, smaller
claim than the headline.